# Proyecto Etapa 3: Métricas de calidad de resultados

**Nombre solicitado por la rúbrica:** Proyecto Etapa3 MetricasCalidadResultados  
**Entregable:** Etapa_3 Equipo#12  
**Materia:** Análisis de grandes volúmenes de datos  
**Profesores:** Iván Olmos Pineda y Luis Daniel Mendoza  
**Equipo 12**

- Carlos Eduardo Vega Campos (A01797803)
- Marco Emilio Jiménez Jiménez (A01797948)
- Martha Alicia Villalobos Facundo (A01840063)
- Jonathan Javier Monsalve Giraldo (A01840272)

## Objetivo

Evaluar modelos de aprendizaje supervisado y no supervisado en PySpark sobre la muestra representativa M del proyecto, usando métricas adecuadas para grandes volúmenes de datos y una partición train-test estratificada.

## 0. Configuración del entorno

Esta sección prepara el entorno local o Colab, inicializa Spark y define constantes globales. Las rutas son relativas al notebook: los datos se esperan en `data/raw`.

In [1]:
import importlib.util
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # En Colab, Java suele estar disponible. Esta verificación deja explícito el requisito.
    java_check = subprocess.run(["bash", "-lc", "java -version"], capture_output=True, text=True)
    if java_check.returncode != 0:
        subprocess.check_call(["apt-get", "update"])
        subprocess.check_call(["apt-get", "install", "-y", "openjdk-11-jdk-headless"])

ensure_package("findspark")
ensure_package("pyspark")
ensure_package("pandas")

import findspark
findspark.init()

import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.clustering import GaussianMixture, KMeans
from pyspark.ml.evaluation import ClusteringEvaluator, RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

SEED_M = 42
SPLIT_SEED = 123
TRAIN_RATIO = 0.80
N_M_TARGET = 5_000_000
MIN_PER_STRATUM = 500
EXPECTED_M = 5_030_141

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
YEARS = (2024, 2025)
DATA_DIR = Path("data") / "raw"

spark = (
    SparkSession.builder
    .appName("Proyecto_Etapa3_MetricasCalidadResultados")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.debug.maxToStringFields", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Entorno Colab: {IN_COLAB}")
print(f"Ruta de datos: {DATA_DIR.resolve()}")
print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 22:29:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Entorno Colab: False
Ruta de datos: /Users/jj/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/materias/3 trimestre/Análisis de grandes volúmenes de datos/workspace/proyecto/etapa3/data/raw
Spark version: 4.1.1


## 1 Construcción de la muestra M

Esta sección reconstruye la muestra M definida en Etapa 2. La mejora respecto a las semanas 5 y 6 es usar M completa, no la submuestra M'. La referencia de comparación es la población analítica posterior a la limpieza cerrada en Etapa 2.

### 1.1 Descarga reproducible de datos

La celda siguiente descarga los 24 archivos Parquet de 2024-2025 y el catálogo de zonas si no existen localmente. La descarga es idempotente: si el archivo ya está en `data/raw`, no se vuelve a descargar.

In [2]:
def fetch(url, target):
    target = Path(target)
    if target.exists():
        return "skip"
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, target)
    return "downloaded"


download_log = []
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        status = fetch(f"{CDN_BASE}/trip-data/{filename}", DATA_DIR / filename)
        download_log.append((filename, status))

zones_status = fetch(f"{CDN_BASE}/misc/taxi_zone_lookup.csv", DATA_DIR / "taxi_zone_lookup.csv")

print("Archivos Parquet revisados:", len(download_log))
print("Archivos descargados:", sum(1 for _, status in download_log if status == "downloaded"))
print("Catálogo de zonas:", zones_status)

Archivos Parquet revisados: 24
Archivos descargados: 0
Catálogo de zonas: skip


### 1.2 Carga, downcast y limpieza cerrada

Se cargan los Parquet con `mergeSchema=True` para conservar columnas introducidas en 2025. Después se aplica el downcast de tipos y los filtros destructivos definidos en Etapa 2. Estos filtros no se rediseñan en esta etapa.

In [3]:
parquet_paths = sorted(str(path) for path in DATA_DIR.glob("yellow_tripdata_*.parquet"))
assert len(parquet_paths) == 24, f"Se esperaban 24 archivos Parquet, se encontraron {len(parquet_paths)}"

df_native = spark.read.option("mergeSchema", "true").parquet(*parquet_paths)
zones = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv"))
)

df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

df_filtered = (
    df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance").between(0, 200))
    .filter(F.col("fare_amount").between(0, 1000))
    .filter(F.col("total_amount").between(0, 1200))
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0)))
)

n_raw = df_raw.count()
n_filtered = df_filtered.count()
pct_removed = (n_raw - n_filtered) / n_raw * 100

print(f"Filas crudas: {n_raw:,}")
print(f"Filas tras limpieza: {n_filtered:,}")
print(f"Pérdida por filtros cerrados de Etapa 2: {pct_removed:.2f}%")

assert n_raw == 89_892_322, "El conteo crudo no coincide con la referencia de Etapa 1"
assert abs(pct_removed - 6.07) < 0.20, "La pérdida por limpieza difiere de la referencia de Etapa 2"

Filas crudas: 89,892,322
Filas tras limpieza: 84,437,138
Pérdida por filtros cerrados de Etapa 2: 6.07%


### 1.3 Imputaciones operativas

Las imputaciones replican Etapa 2: se corrigen nulos estructurales y valores fuera de dominio sin descartar filas adicionales. La bandera `is_flex_fare` se conserva para distinguir el régimen Flex.

In [4]:
df_clean = (
    df_filtered
    .withColumn(
        "passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
        .otherwise(F.lit(1).cast("byte")),
    )
    .withColumn(
        "cbd_congestion_fee",
        F.when(
            F.col("cbd_congestion_fee").isNull()
            | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
            F.lit(0.0).cast("float"),
        ).otherwise(F.col("cbd_congestion_fee")),
    )
    .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee", F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), F.lit("F")))
)

imputed_cols = [
    "passenger_count",
    "cbd_congestion_fee",
    "congestion_surcharge",
    "Airport_fee",
    "RatecodeID",
    "store_and_fwd_flag",
]

null_row = df_clean.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols]).first()
null_summary = [(c, int(null_row[c] or 0)) for c in imputed_cols]

print("Nulos remanentes tras imputación:")
for col_name, n_nulls in null_summary:
    print(f"{col_name}: {n_nulls:,}")

assert all(n_nulls == 0 for _, n_nulls in null_summary), "Quedan nulos en columnas imputadas"

Nulos remanentes tras imputación:
passenger_count: 0
cbd_congestion_fee: 0
congestion_surcharge: 0
Airport_fee: 0
RatecodeID: 0
store_and_fwd_flag: 0


### 1.4 Variables de caracterización

La muestra M se estratifica con cuatro variables: macrozona de origen, grupo de pago, bloque horario y rango de distancia. Su concatenación forma `stratum_id`, la unidad de muestreo y partición.

In [5]:
airport_ids = {row.LocationID for row in zones.filter(F.col("service_zone").isin("Airports", "EWR")).collect()}
unknown_ids = {264, 265}
manhattan_ids = {row.LocationID for row in zones.filter(F.col("Borough") == "Manhattan").collect()} - airport_ids - unknown_ids
outer_ids = {row.LocationID for row in zones.filter(F.col("Borough").isin("Brooklyn", "Queens", "Bronx", "Staten Island")).collect()} - airport_ids - unknown_ids

assert not (airport_ids & manhattan_ids)
assert not (airport_ids & outer_ids)
assert not (unknown_ids & manhattan_ids)
assert not (unknown_ids & outer_ids)

df_feat = (
    df_clean
    .withColumn(
        "pu_macro_zone",
        F.when(F.col("PULocationID").isin(sorted(airport_ids)), "airport")
        .when(F.col("PULocationID").isin(sorted(unknown_ids)), "unknown")
        .when(F.col("PULocationID").isin(sorted(manhattan_ids)), "manhattan")
        .when(F.col("PULocationID").isin(sorted(outer_ids)), "outer_borough")
        .otherwise("unknown"),
    )
    .withColumn(
        "payment_group",
        F.when(F.col("payment_type") == 0, "flex")
        .when(F.col("payment_type") == 1, "credit")
        .when(F.col("payment_type") == 2, "cash")
        .otherwise("other"),
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow", F.dayofweek("tpep_pickup_datetime"))
    .withColumn(
        "day_hour_bucket",
        F.when(F.col("pickup_hour").between(0, 5), "late_night")
        .when(F.col("dow").isin(1, 7), "weekend")
        .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(6, 10), "weekday_am")
        .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(16, 20), "weekday_pm_peak")
        .otherwise("other"),
    )
    .withColumn(
        "trip_distance_bin",
        F.when(F.col("trip_distance") < 1.12, "short")
        .when(F.col("trip_distance") < 12.43, "medium")
        .otherwise("long"),
    )
    .withColumn("is_flex_fare", F.col("payment_type") == 0)
    .withColumn(
        "cbd_period_flag",
        F.when(F.col("tpep_pickup_datetime") < F.lit("2025-01-05"), "pre_cbd")
        .otherwise("post_cbd"),
    )
    .withColumn(
        "stratum_id",
        F.concat_ws(
            "|",
            F.col("pu_macro_zone"),
            F.col("payment_group"),
            F.col("day_hour_bucket"),
            F.col("trip_distance_bin"),
        ),
    )
)

characterization_cols = ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]
expected_values = {
    "pu_macro_zone": {"airport", "manhattan", "outer_borough", "unknown"},
    "payment_group": {"flex", "credit", "cash", "other"},
    "day_hour_bucket": {"late_night", "weekend", "weekday_am", "weekday_pm_peak", "other"},
    "trip_distance_bin": {"short", "medium", "long"},
}

for col_name, expected in expected_values.items():
    observed = {row[col_name] for row in df_feat.select(col_name).distinct().collect()}
    assert observed == expected, f"{col_name}: valores observados {observed}, esperados {expected}"

print("Variables de caracterización construidas y validadas.")

Variables de caracterización construidas y validadas.


### 1.5 Fracciones de muestreo

Se calculan las fracciones por estrato con el tamaño objetivo de 5,000,000 filas y piso mínimo de 500. El piso protege combinaciones raras para que sigan presentes en M.

In [6]:
n_population = df_feat.count()

D_dist = (
    df_feat.groupBy("stratum_id")
    .count()
    .withColumnRenamed("count", "n_D")
    .withColumn("p_D", F.col("n_D") / F.lit(n_population))
)

n_strata_D = D_dist.count()
print(f"Estratos poblados en población analítica: {n_strata_D} de 240")
assert n_strata_D == 240, "Se esperaban 240 estratos poblados"

target = (
    D_dist
    .withColumn("target_n_raw", F.round(F.lit(N_M_TARGET) * F.col("p_D")).cast("long"))
    .withColumn(
        "target_n",
        F.least(
            F.col("n_D"),
            F.greatest(F.lit(MIN_PER_STRATUM).cast("long"), F.col("target_n_raw")),
        ),
    )
    .withColumn("fraction", F.col("target_n") / F.col("n_D"))
)

fractions_etapa2 = {
    row["stratum_id"]: float(row["fraction"])
    for row in target.select("stratum_id", "fraction").collect()
}

assert len(fractions_etapa2) == 240
assert all(0 < value <= 1.0 for value in fractions_etapa2.values())

expected_target_n = target.agg(F.sum("target_n").alias("n")).first()["n"]
print(f"Tamaño esperado de M por fracciones: {expected_target_n:,}")

Estratos poblados en población analítica: 240 de 240


Tamaño esperado de M por fracciones: 5,030,141


### 1.6 Extracción y validación de M

Se aplica `sampleBy` con semilla fija para reconstruir M. Después se compara la proporción de cada estrato frente a la población analítica.

In [7]:
M = df_feat.stat.sampleBy("stratum_id", fractions_etapa2, seed=SEED_M).cache()
n_M = M.count()

M_dist = (
    M.groupBy("stratum_id")
    .count()
    .withColumnRenamed("count", "n_M")
    .withColumn("p_M", F.col("n_M") / F.lit(n_M))
)

n_strata_M = M_dist.count()
print(f"Tamaño real de M: {n_M:,}")
print(f"Estratos poblados en M: {n_strata_M} de 240")

assert abs(n_M - EXPECTED_M) / EXPECTED_M < 0.02, "M difiere más de 2% del tamaño esperado"
assert n_strata_M == 240, "M debe conservar los 240 estratos"

repr_check = (
    D_dist.join(M_dist, "stratum_id", "inner")
    .withColumn("diff_pp", (F.col("p_M") - F.col("p_D")) * 100)
    .withColumn("abs_diff_pp", F.abs(F.col("diff_pp")))
)

repr_check.orderBy(F.desc("abs_diff_pp")).show(10, truncate=False)

Tamaño real de M: 5,029,725
Estratos poblados en M: 240 de 240


+---------------------------------------+--------+--------------------+------+--------------------+---------------------+--------------------+
|stratum_id                             |n_D     |p_D                 |n_M   |p_M                 |diff_pp              |abs_diff_pp         |
+---------------------------------------+--------+--------------------+------+--------------------+---------------------+--------------------+
|manhattan|credit|other|medium          |11515883|0.13638409913893576 |681296|0.13545392640750736 |-0.0930172731428397  |0.0930172731428397  |
|manhattan|credit|weekday_pm_peak|medium|9088533 |0.10763667759558596 |537414|0.10684759107108241 |-0.07890865245035461 |0.07890865245035461 |
|manhattan|credit|weekend|medium        |8179141 |0.09686662994191016 |483828|0.09619372828534363 |-0.06729016565665269 |0.06729016565665269 |
|manhattan|credit|other|short           |5434482 |0.06436127666951479 |321013|0.06382317124693695 |-0.05381054225778309 |0.05381054225778309 |

### 1.7 Representatividad marginal

La tabla siguiente resume la representatividad de M en las cuatro variables de caracterización. Las diferencias se interpretan en puntos porcentuales.

In [8]:
def collect_distribution(df, column_name, total_rows):
    return {
        row[column_name]: row["count"] / total_rows
        for row in df.groupBy(column_name).count().collect()
    }


representativity_rows = []
for column_name in characterization_cols:
    p_population = collect_distribution(df_feat, column_name, n_population)
    p_sample = collect_distribution(M, column_name, n_M)
    for category in sorted(set(p_population) | set(p_sample)):
        p_d = p_population.get(category, 0.0)
        p_m = p_sample.get(category, 0.0)
        representativity_rows.append({
            "variable": column_name,
            "categoria": str(category),
            "pct_poblacion_analitica": round(p_d * 100, 4),
            "pct_M": round(p_m * 100, 4),
            "diff_pp_M_menos_P": round((p_m - p_d) * 100, 4),
        })

representativity_pdf = pd.DataFrame(representativity_rows)
display(representativity_pdf)

,variable,categoria,pct_poblacion_analitica,pct_M,diff_pp_M_menos_P
0,pu_macro_zone,airport,7.4263,7.5341,0.1078
1,pu_macro_zone,manhattan,87.4987,86.9657,-0.5330
2,pu_macro_zone,outer_borough,4.8288,4.9506,0.1218
3,pu_macro_zone,unknown,0.2463,0.5496,0.3034
4,payment_group,cash,11.4359,11.5382,0.1023
5,payment_group,credit,72.2234,71.8373,-0.3861
6,payment_group,flex,14.8591,14.9177,0.0586
7,payment_group,other,1.4816,1.7069,0.2253
8,day_hour_bucket,late_night,8.4615,8.5653,0.1037
9,day_hour_bucket,other,31.2623,31.1564,-0.1059


**Lectura de la sección 1**

La reconstrucción de la muestra parte de 89,892,322 viajes crudos y conserva 84,437,138 después de aplicar los filtros de calidad cerrados en Etapa 2, una pérdida de 6.07%. Esta pérdida coincide con la referencia previa, por lo que la limpieza no reabre decisiones metodológicas anteriores.

La población analítica mantiene 240 de 240 estratos poblados. Con las fracciones de Etapa 2, el tamaño esperado de M era 5,030,141 filas y el tamaño real obtenido fue 5,029,725 filas, con los 240 estratos representados. La diferencia frente al tamaño esperado es mínima y entra dentro de la variación normal de `sampleBy`.

La comparación de marginales M vs población analítica muestra desviaciones pequeñas en las variables de caracterización. La mayor diferencia es Manhattan, con -0.5330 puntos porcentuales, seguida por crédito con -0.3861 pp y zona desconocida con +0.3034 pp. La sobre-representación de categorías raras es esperada porque el diseño de Etapa 2 protege estratos pequeños con un piso mínimo.

La mejora de esta etapa no es que toda métrica vaya a mejorar automáticamente, sino que la evaluación se vuelve más exigente y estable: M conserva todos los estratos con soporte amplio y evita depender de M', una reducción de aproximadamente 20% usada en semanas 5 y 6 para facilitar el cómputo individual.

## 2 Construcción Train - Test

Esta sección divide M en entrenamiento y prueba con ventana exacta por `stratum_id`. La partición conserva la probabilidad de ocurrencia de los patrones de caracterización.

In [9]:
counts_M = M.groupBy("stratum_id").count().withColumnRenamed("count", "n_M_s")
w_split = Window.partitionBy("stratum_id").orderBy(F.rand(seed=SPLIT_SEED))

M_split = (
    M.join(counts_M, "stratum_id")
    .withColumn("rn", F.row_number().over(w_split))
    .withColumn("split_id", F.concat_ws("|", F.col("stratum_id"), F.col("rn").cast("string")))
    .withColumn("train_cutoff", F.floor(F.lit(TRAIN_RATIO) * F.col("n_M_s")).cast("long"))
    .cache()
)

train_df = (
    M_split.filter(F.col("rn") <= F.col("train_cutoff"))
    .drop("rn", "train_cutoff", "n_M_s")
    .cache()
)

test_df = (
    M_split.filter(F.col("rn") > F.col("train_cutoff"))
    .drop("rn", "train_cutoff", "n_M_s")
    .cache()
)

n_train = train_df.count()
n_test = test_df.count()

print(f"Filas train: {n_train:,}")
print(f"Filas test: {n_test:,}")
print(f"Proporción train: {n_train / (n_train + n_test):.4f}")

assert n_train + n_test == n_M, "Train + test debe reconstruir M"

Filas train: 4,023,683
Filas test: 1,006,042
Proporción train: 0.8000


In [10]:
intersection_count = train_df.select("split_id").intersect(test_df.select("split_id")).count()
union_distinct_count = train_df.unionByName(test_df).select("split_id").distinct().count()

print(f"Intersección train-test por split_id: {intersection_count:,}")
print(f"Distinct split_id en unión train-test: {union_distinct_count:,}")

assert intersection_count == 0, "Train y test no deben solaparse"
assert union_distinct_count == n_M, "La unión train-test debe reconstruir M"

strata_train = train_df.select("stratum_id").distinct().count()
strata_test = test_df.select("stratum_id").distinct().count()
min_train = train_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]
min_test = test_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"Estratos en train: {strata_train}")
print(f"Estratos en test: {strata_test}")
print(f"Mínimo por estrato en train: {min_train}")
print(f"Mínimo por estrato en test: {min_test}")

assert strata_train == 240 and strata_test == 240, "Train y test deben conservar 240 estratos"

Intersección train-test por split_id: 0
Distinct split_id en unión train-test: 5,029,725
Estratos en train: 240
Estratos en test: 240
Mínimo por estrato en train: 42
Mínimo por estrato en test: 11


In [11]:
split_balance_rows = []
for column_name in characterization_cols:
    p_train = collect_distribution(train_df, column_name, n_train)
    p_test = collect_distribution(test_df, column_name, n_test)
    for category in sorted(set(p_train) | set(p_test)):
        tr = p_train.get(category, 0.0)
        ts = p_test.get(category, 0.0)
        split_balance_rows.append({
            "variable": column_name,
            "categoria": str(category),
            "pct_train": round(tr * 100, 4),
            "pct_test": round(ts * 100, 4),
            "diff_pp_train_menos_test": round((tr - ts) * 100, 4),
        })

split_balance_pdf = pd.DataFrame(split_balance_rows)
display(split_balance_pdf)

,variable,categoria,pct_train,pct_test,diff_pp_train_menos_test
0,pu_macro_zone,airport,7.5336,7.5359,-0.0022
1,pu_macro_zone,manhattan,86.9672,86.9597,0.0075
2,pu_macro_zone,outer_borough,4.9501,4.9527,-0.0026
3,pu_macro_zone,unknown,0.5491,0.5518,-0.0027
4,payment_group,cash,11.5377,11.5399,-0.0021
5,payment_group,credit,71.8384,71.8328,0.0056
6,payment_group,flex,14.9175,14.9185,-0.0010
7,payment_group,other,1.7064,1.7089,-0.0025
8,day_hour_bucket,late_night,8.5650,8.5664,-0.0015
9,day_hour_bucket,other,31.1567,31.1554,0.0013


**Lectura de la sección 2**

La partición 80/20 generó 4,023,683 filas de entrenamiento y 1,006,042 filas de prueba, con proporción train igual a 0.8000. La verificación con `split_id` confirma que `Tr ∩ Ts = vacío`: la intersección tiene 0 filas y la unión reconstruye los 5,029,725 registros de M.

Tanto train como test conservan los 240 estratos. El estrato más pequeño queda con 42 filas en train y 11 en test, por lo que incluso los perfiles raros protegidos por el piso de Etapa 2 siguen presentes en ambos conjuntos.

Las diferencias marginales train-test son prácticamente nulas: la mayor desviación observada es 0.0075 puntos porcentuales en `pu_macro_zone = manhattan`. Esto confirma que la ventana exacta por `stratum_id` conserva la probabilidad de ocurrencia de los patrones y evita el sesgo que podría introducir un split aleatorio simple.

## 3 Selección de métricas para medir calidad de resultados

Las métricas se fijan antes del entrenamiento para evitar seleccionar indicadores después de observar resultados.

| Paradigma | Métrica | Qué mide | Uso en esta etapa | Limitación en Big Data |
|---|---|---|---|---|
| Supervisado, regresión | RMSE | Error cuadrático medio en unidades de USD | Métrica principal para penalizar errores grandes en `fare_amount` | Agregación distribuida barata, pero sensible a outliers |
| Supervisado, regresión | MAE | Error absoluto medio en USD | Interpretación directa del error típico | Agregación distribuida barata, pero no penaliza la cola tanto como RMSE |
| Supervisado, regresión | R² | Varianza explicada por el modelo | Comparación global entre baseline y modelo principal | Depende de la varianza del conjunto evaluado |
| Supervisado, regresión | MAE/RMSE por `is_flex_fare` | Error por régimen tarifario | Evalúa si el modelo generaliza igual en Metered y Flex Fare | Requiere suficiente soporte por segmento |
| No supervisado, clustering | Silhouette | Cohesión y separación de clusters | Comparación de KMeans y GMM en train y test | Costosa sobre vectores grandes, aunque Spark la calcula de forma distribuida |
| No supervisado, clustering | `trainingCost` | Suma de distancias cuadradas intra-cluster en KMeans | Apoya selección de k | Baja al aumentar k, por lo que no basta sola |
| No supervisado, clustering | Balance de clusters | Fracción del cluster dominante y tamaños mínimos | Evita aceptar particiones con un grupo dominante | No mide separación geométrica |
| No supervisado, GMM | Log-likelihood de train | Ajuste probabilístico del modelo | Diagnóstico del ajuste GMM | Spark no expone log-likelihood de test sin código adicional |
| No supervisado, GMM | Probabilidad máxima promedio | Confianza de asignación a componentes | Lectura de certeza de pertenencia | Puede ser alta aunque los clusters sean poco útiles operativamente |

## 4 Entrenamiento de Modelos de Aprendizaje

Esta sección implementa los modelos definidos: regresión supervisada de `fare_amount` y clustering no supervisado de arquetipos operativos. El entrenamiento usa solo Tr; Ts se reserva para evaluación.

In [12]:
def add_trip_duration(df):
    return (
        df
        .withColumn(
            "trip_duration_min_raw",
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / F.lit(60.0),
        )
        .filter(F.col("trip_duration_min_raw").between(0.5, 720.0))
        .withColumn("trip_duration_min", F.col("trip_duration_min_raw").cast("float"))
        .drop("trip_duration_min_raw")
    )


train_base_ml = add_trip_duration(train_df).withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte")).cache()
test_base_ml = add_trip_duration(test_df).withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte")).cache()

n_train_base_ml = train_base_ml.count()
n_test_base_ml = test_base_ml.count()

print(f"Train antes de filtro de duración: {n_train:,}")
print(f"Test antes de filtro de duración: {n_test:,}")
print(f"Train para modelado: {n_train_base_ml:,}")
print(f"Test para modelado: {n_test_base_ml:,}")
print(f"Pérdida train por duración: {(n_train - n_train_base_ml) / n_train * 100:.3f}%")
print(f"Pérdida test por duración: {(n_test - n_test_base_ml) / n_test * 100:.3f}%")

Train antes de filtro de duración: 4,023,683
Test antes de filtro de duración: 1,006,042
Train para modelado: 3,986,017
Test para modelado: 996,458
Pérdida train por duración: 0.936%
Pérdida test por duración: 0.953%


### 4.1 Track supervisado: regresión de `fare_amount`

Se entrenan dos modelos: `LinearRegression` como baseline y `RandomForestRegressor` como modelo principal. Las columnas de cobro derivadas se excluyen para evitar fuga de información.

**Recap breve de Semana 5**

En Semana 5 se formuló el problema como regresión de `fare_amount`, con `LinearRegression` como baseline y `RandomForestRegressor` como modelo principal. Se conservaron las mismas 8 features y la misma lista de columnas excluidas por fuga. La diferencia de esta etapa es que el entrenamiento y la evaluación usan M completa, no M'.

In [13]:
TARGET = "fare_amount"
SUPERVISED_CAT_COLS = ["pu_macro_zone", "RatecodeID", "day_hour_bucket", "cbd_period_flag"]
SUPERVISED_NUM_COLS = ["trip_distance", "trip_duration_min", "is_flex_fare", "passenger_count"]
SUPERVISED_FEATURE_COLS = [
    "trip_distance",
    "trip_duration_min",
    "pu_macro_zone",
    "RatecodeID",
    "is_flex_fare",
    "day_hour_bucket",
    "cbd_period_flag",
    "passenger_count",
]
LEAKAGE_COLS = {
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
}

assert set(SUPERVISED_FEATURE_COLS) == set(SUPERVISED_CAT_COLS + SUPERVISED_NUM_COLS)
assert TARGET not in SUPERVISED_FEATURE_COLS
assert not (set(SUPERVISED_FEATURE_COLS) & LEAKAGE_COLS), "Hay columnas con fuga en las features"

missing_supervised = sorted(set(SUPERVISED_FEATURE_COLS + [TARGET]) - set(train_base_ml.columns))
assert not missing_supervised, f"Columnas faltantes para supervisado: {missing_supervised}"

supervised_indexers = [
    StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    for col_name in SUPERVISED_CAT_COLS
]
supervised_encoder = OneHotEncoder(
    inputCols=[f"{col_name}_idx" for col_name in SUPERVISED_CAT_COLS],
    outputCols=[f"{col_name}_ohe" for col_name in SUPERVISED_CAT_COLS],
)
supervised_assembler = VectorAssembler(
    inputCols=SUPERVISED_NUM_COLS + [f"{col_name}_ohe" for col_name in SUPERVISED_CAT_COLS],
    outputCol="features",
    handleInvalid="keep",
)

supervised_preprocessing = supervised_indexers + [supervised_encoder, supervised_assembler]

ev_rmse = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="rmse")
ev_mae = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="mae")
ev_r2 = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="r2")

print("Features supervisadas:", SUPERVISED_FEATURE_COLS)
print("Columnas excluidas por fuga:", sorted(LEAKAGE_COLS))

Features supervisadas: ['trip_distance', 'trip_duration_min', 'pu_macro_zone', 'RatecodeID', 'is_flex_fare', 'day_hour_bucket', 'cbd_period_flag', 'passenger_count']
Columnas excluidas por fuga: ['Airport_fee', 'cbd_congestion_fee', 'congestion_surcharge', 'extra', 'improvement_surcharge', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount']


In [14]:
lr = LinearRegression(
    featuresCol="features",
    labelCol=TARGET,
    regParam=0.0,
    elasticNetParam=0.0,
    maxIter=100,
)

pipeline_lr = Pipeline(stages=supervised_preprocessing + [lr])

t0 = time.time()
model_lr = pipeline_lr.fit(train_base_ml)
lr_fit_seconds = time.time() - t0

pred_lr = model_lr.transform(test_base_ml).cache()

metrics_lr = {
    "modelo": "LinearRegression",
    "rmse": ev_rmse.evaluate(pred_lr),
    "mae": ev_mae.evaluate(pred_lr),
    "r2": ev_r2.evaluate(pred_lr),
    "fit_seconds": lr_fit_seconds,
}

print(metrics_lr)

26/06/12 22:31:35 WARN Instrumentation: [1b43dd73] regParam is zero, which might cause numerical instability and overfitting.
26/06/12 22:31:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/12 22:31:36 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/06/12 22:31:36 WARN Instrumentation: [1b43dd73] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


{'modelo': 'LinearRegression', 'rmse': 5.595138572242773, 'mae': 2.453751991817976, 'r2': 0.900296892097442, 'fit_seconds': 4.573385000228882}


In [15]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=TARGET,
    featureSubsetStrategy="auto",
    seed=SEED_M,
)

pipeline_rf = Pipeline(stages=supervised_preprocessing + [rf])

param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 50])
    .addGrid(rf.maxDepth, [6, 8])
    .addGrid(rf.subsamplingRate, [1.0])
    .build()
)

tvs = TrainValidationSplit(
    estimator=pipeline_rf,
    estimatorParamMaps=param_grid,
    evaluator=ev_rmse,
    trainRatio=0.75,
    parallelism=2,
    seed=SEED_M,
)

t0 = time.time()
tvs_model = tvs.fit(train_base_ml)
rf_fit_seconds = time.time() - t0

validation_rows = []
for params, rmse_value in zip(param_grid, tvs_model.validationMetrics):
    validation_rows.append({param.name: value for param, value in params.items()} | {"rmse_validacion": float(rmse_value)})

validation_results_pdf = pd.DataFrame(validation_rows).sort_values("rmse_validacion")
display(validation_results_pdf)

best_pipeline_rf = tvs_model.bestModel
pred_rf = best_pipeline_rf.transform(test_base_ml).cache()

metrics_rf = {
    "modelo": "RandomForestRegressor",
    "rmse": ev_rmse.evaluate(pred_rf),
    "mae": ev_mae.evaluate(pred_rf),
    "r2": ev_r2.evaluate(pred_rf),
    "fit_seconds": rf_fit_seconds,
}

print(metrics_rf)

26/06/12 22:32:09 WARN DAGScheduler: Broadcasting large task binary with size 1443.9 KiB
26/06/12 22:32:50 WARN DAGScheduler: Broadcasting large task binary with size 1275.6 KiB
26/06/12 22:32:56 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/06/12 22:33:32 WARN DAGScheduler: Broadcasting large task binary with size 1266.0 KiB
26/06/12 22:33:39 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


,numTrees,maxDepth,subsamplingRate,rmse_validacion
3,50,8,1.0,5.246516
1,30,8,1.0,5.263093
2,50,6,1.0,5.682404
0,30,6,1.0,5.714892


{'modelo': 'RandomForestRegressor', 'rmse': 5.114245145377588, 'mae': 1.688993698655895, 'r2': 0.9166990248571845, 'fit_seconds': 127.98242998123169}


In [16]:
supervised_results_pdf = pd.DataFrame([metrics_lr, metrics_rf])
display(supervised_results_pdf)

segment_rows = []
for segment_name, segment_value in [("Metered", 0), ("Flex Fare", 1)]:
    segment_pred = pred_rf.filter(F.col("is_flex_fare") == segment_value).cache()
    segment_n = segment_pred.count()
    segment_rows.append({
        "segmento": segment_name,
        "n": segment_n,
        "rmse": ev_rmse.evaluate(segment_pred),
        "mae": ev_mae.evaluate(segment_pred),
    })
    segment_pred.unpersist()

rf_segment_results_pdf = pd.DataFrame(segment_rows)
display(rf_segment_results_pdf)

pred_lr.unpersist()
pred_rf.unpersist()

,modelo,rmse,mae,r2,fit_seconds
0,LinearRegression,5.595139,2.453752,0.900297,4.573385
1,RandomForestRegressor,5.114245,1.688994,0.916699,127.982430


,segmento,n,rmse,mae
0,Metered,846601,4.456311,1.175662
1,Flex Fare,149857,7.856755,4.589008


DataFrame[stratum_id: string, VendorID: tinyint, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: tinyint, trip_distance: float, RatecodeID: tinyint, store_and_fwd_flag: string, PULocationID: smallint, DOLocationID: smallint, payment_type: tinyint, fare_amount: float, extra: float, mta_tax: float, tip_amount: float, tolls_amount: float, improvement_surcharge: float, total_amount: float, congestion_surcharge: float, Airport_fee: float, cbd_congestion_fee: float, pu_macro_zone: string, payment_group: string, pickup_hour: int, dow: int, day_hour_bucket: string, trip_distance_bin: string, is_flex_fare: tinyint, cbd_period_flag: string, split_id: string, trip_duration_min: float, pu_macro_zone_idx: double, RatecodeID_idx: double, day_hour_bucket_idx: double, cbd_period_flag_idx: double, pu_macro_zone_ohe: vector, RatecodeID_ohe: vector, day_hour_bucket_ohe: vector, cbd_period_flag_ohe: vector, features: vector, prediction: double]

**Lectura del track supervisado**

El modelo principal es `RandomForestRegressor`. En test, `LinearRegression` obtuvo RMSE 5.5951, MAE 2.4538 y R² 0.9003. El bosque aleatorio redujo el error a RMSE 5.1142 y MAE 1.6890, con R² 0.9167. La mejora relativa del bosque es aproximadamente 8.6% en RMSE y 31.2% en MAE, aunque con mayor tiempo de ajuste.

La mejor configuración en la rejilla acotada fue `numTrees = 50`, `maxDepth = 8` y `subsamplingRate = 1.0`, con RMSE de validación 5.2465. Esto coincide con la experiencia de la semana 5: una profundidad mayor captura mejor no linealidades e interacciones entre distancia, duración, zona y régimen tarifario.

El desglose por régimen tarifario anticipa el punto crítico del análisis final: el modelo funciona mejor para viajes Metered que para Flex Fare. En Metered, el bosque aleatorio obtiene MAE 1.1757 y RMSE 4.4563; en Flex Fare, el MAE sube a 4.5890 y el RMSE a 7.8568.


### 4.2 Track no supervisado: clustering de arquetipos operativos

Se entrena KMeans con un barrido pequeño de k y GaussianMixture como comparación probabilística. Las variables monetarias no entran al vector de clustering.

**Recap breve de Semana 6**

En Semana 6 el problema no supervisado se definió como segmentación de arquetipos operativos de viaje. Se reutiliza el mismo vector de 8 features: distancia, duración, pasajeros, velocidad media capada, macrozona, RatecodeID, bloque horario e indicador Flex. Se excluye `cbd_period_flag` del entrenamiento para evitar que el modelo se convierta principalmente en una separación calendario/regulación.

La velocidad media se capa en 40 mph para evitar que viajes con duración o distancia anómalas dominen la geometría de KMeans y GMM. El valor se mantiene de Semana 6 porque queda por encima del rango operativo típico y funciona como protección contra colas extremas sin eliminar la señal de viajes rápidos.

In [17]:
AVERAGE_SPEED_CAP_MPH = 40.0

train_cluster_ml = (
    train_base_ml
    .withColumn("average_speed_mph", F.col("trip_distance") / (F.col("trip_duration_min") / F.lit(60.0)))
    .withColumn("average_speed_mph_capped", F.least(F.col("average_speed_mph"), F.lit(AVERAGE_SPEED_CAP_MPH)))
    .cache()
)

test_cluster_ml = (
    test_base_ml
    .withColumn("average_speed_mph", F.col("trip_distance") / (F.col("trip_duration_min") / F.lit(60.0)))
    .withColumn("average_speed_mph_capped", F.least(F.col("average_speed_mph"), F.lit(AVERAGE_SPEED_CAP_MPH)))
    .cache()
)

CLUSTER_NUM_COLS = ["trip_distance", "trip_duration_min", "passenger_count", "average_speed_mph_capped"]
CLUSTER_CAT_COLS = ["pu_macro_zone", "RatecodeID", "day_hour_bucket"]
CLUSTER_BIN_COLS = ["is_flex_fare"]
CLUSTER_FEATURE_COLS = CLUSTER_NUM_COLS + CLUSTER_CAT_COLS + CLUSTER_BIN_COLS
PROFILE_ONLY_COLS = ["cbd_period_flag", "average_speed_mph"]

EXCLUDED_CLUSTER_AMOUNT_COLS = {
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
}

assert len(CLUSTER_FEATURE_COLS) == 8
assert "cbd_period_flag" not in CLUSTER_FEATURE_COLS
assert not (set(CLUSTER_FEATURE_COLS) & EXCLUDED_CLUSTER_AMOUNT_COLS)

cluster_indexers = [
    StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    for col_name in CLUSTER_CAT_COLS
]
cluster_encoder = OneHotEncoder(
    inputCols=[f"{col_name}_idx" for col_name in CLUSTER_CAT_COLS],
    outputCols=[f"{col_name}_ohe" for col_name in CLUSTER_CAT_COLS],
)
num_assembler = VectorAssembler(inputCols=CLUSTER_NUM_COLS, outputCol="num_features", handleInvalid="keep")
scaler = StandardScaler(inputCol="num_features", outputCol="scaled_num_features", withMean=True, withStd=True)
cluster_assembler = VectorAssembler(
    inputCols=["scaled_num_features"] + [f"{col_name}_ohe" for col_name in CLUSTER_CAT_COLS] + CLUSTER_BIN_COLS,
    outputCol="features",
    handleInvalid="keep",
)

cluster_preprocess_pipeline = Pipeline(stages=cluster_indexers + [cluster_encoder, num_assembler, scaler, cluster_assembler])
cluster_preprocess_model = cluster_preprocess_pipeline.fit(train_cluster_ml)

train_features = cluster_preprocess_model.transform(train_cluster_ml).cache()
test_features = cluster_preprocess_model.transform(test_cluster_ml).cache()

n_train_features = train_features.count()
n_test_features = test_features.count()

print(f"Features clustering ({len(CLUSTER_FEATURE_COLS)}): {CLUSTER_FEATURE_COLS}")
print(f"Filas con features train: {n_train_features:,}")
print(f"Filas con features test: {n_test_features:,}")

Features clustering (8): ['trip_distance', 'trip_duration_min', 'passenger_count', 'average_speed_mph_capped', 'pu_macro_zone', 'RatecodeID', 'day_hour_bucket', 'is_flex_fare']
Filas con features train: 3,986,017
Filas con features test: 996,458


**Diagnóstico del cap de velocidad**

La celda siguiente cuantifica cuántos viajes quedan por encima del cap de 40 mph y reporta percentiles de la velocidad media antes de capar. El objetivo no es elegir un nuevo umbral, sino documentar el impacto de la decisión heredada de Semana 6 sobre M completa.

In [18]:
speed_quantile_probs = [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
speed_quantiles = train_cluster_ml.approxQuantile("average_speed_mph", speed_quantile_probs, 0.001)

speed_cap_rows = []
for prob, value in zip(speed_quantile_probs, speed_quantiles):
    speed_cap_rows.append({"percentil": prob, "average_speed_mph": value})

n_above_speed_cap = train_cluster_ml.filter(F.col("average_speed_mph") > AVERAGE_SPEED_CAP_MPH).count()
pct_above_speed_cap = n_above_speed_cap / n_train_features * 100

print(f"Cap de velocidad media: {AVERAGE_SPEED_CAP_MPH:.1f} mph")
print(f"Viajes train por encima del cap: {n_above_speed_cap:,} ({pct_above_speed_cap:.4f}%)")
display(pd.DataFrame(speed_cap_rows))

Cap de velocidad media: 40.0 mph
Viajes train por encima del cap: 14,279 (0.3582%)


,percentil,average_speed_mph
0,0.500,9.381356
1,0.750,12.964854
2,0.900,19.099820
3,0.950,24.253821
4,0.990,34.513465
5,0.999,4482.857154


In [19]:
K_VALUES = [3, 4, 5, 6]
KMEANS_MAX_ITER = 30

kmeans_evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

kmeans_models = {}
kmeans_rows = []

for k in K_VALUES:
    kmeans = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=k,
        seed=SEED_M,
        maxIter=KMEANS_MAX_ITER,
    )
    model = kmeans.fit(train_features)
    kmeans_models[k] = model

    pred_train_k = model.transform(train_features).select("features", "cluster")
    pred_test_k = model.transform(test_features).select("features", "cluster")

    cluster_sizes = pred_train_k.groupBy("cluster").count().cache()
    size_stats = cluster_sizes.agg(
        F.min("count").alias("min_cluster"),
        F.max("count").alias("max_cluster"),
    ).first()
    cluster_sizes.unpersist()

    kmeans_rows.append({
        "modelo": f"KMeans_k_{k}",
        "k": k,
        "training_cost": float(model.summary.trainingCost),
        "silhouette_train": float(kmeans_evaluator.evaluate(pred_train_k)),
        "silhouette_test": float(kmeans_evaluator.evaluate(pred_test_k)),
        "min_cluster": int(size_stats["min_cluster"]),
        "max_cluster": int(size_stats["max_cluster"]),
        "max_cluster_fraction": float(size_stats["max_cluster"] / n_train_features),
    })

kmeans_results_pdf = pd.DataFrame(kmeans_rows).sort_values("k")
display(kmeans_results_pdf)

,modelo,k,training_cost,silhouette_train,silhouette_test,min_cluster,max_cluster,max_cluster_fraction
0,KMeans_k_3,3,1.282950e+07,0.491448,0.491598,536166,2854210,0.716056
1,KMeans_k_4,4,1.193320e+07,0.375543,0.377448,103425,2853164,0.715793
2,KMeans_k_5,5,1.013852e+07,0.455639,0.455857,199897,2586403,0.648869
3,KMeans_k_6,6,9.747580e+06,0.240329,0.240804,222072,1346659,0.337846


In [20]:
SELECTED_K = 5

kmeans_final = kmeans_models.get(SELECTED_K)
if kmeans_final is None:
    kmeans_final = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=SELECTED_K,
        seed=SEED_M,
        maxIter=KMEANS_MAX_ITER,
    ).fit(train_features)

kmeans_train = kmeans_final.transform(train_features).cache()
kmeans_test = kmeans_final.transform(test_features).cache()

final_kmeans_sil_train = kmeans_evaluator.evaluate(kmeans_train.select("features", "cluster"))
final_kmeans_sil_test = kmeans_evaluator.evaluate(kmeans_test.select("features", "cluster"))

kmeans_counts = (
    kmeans_train.groupBy("cluster")
    .count()
    .withColumn("cluster_fraction", F.round(F.col("count") / F.lit(n_train_features), 4))
    .orderBy("cluster")
)

print(f"KMeans final k={SELECTED_K}")
print(f"Silhouette train: {final_kmeans_sil_train:.4f}")
print(f"Silhouette test: {final_kmeans_sil_test:.4f}")
kmeans_counts.show(truncate=False)

KMeans final k=5
Silhouette train: 0.4556
Silhouette test: 0.4559
+-------+-------+----------------+
|cluster|count  |cluster_fraction|
+-------+-------+----------------+
|0      |542923 |0.1362          |
|1      |199897 |0.0501          |
|2      |2586403|0.6489          |
|3      |423588 |0.1063          |
|4      |233206 |0.0585          |
+-------+-------+----------------+



In [21]:
def top_category_by_cluster(df, col_name, cluster_col="cluster"):
    w_top = Window.partitionBy(cluster_col).orderBy(F.desc("count"), F.asc(col_name))
    return (
        df.groupBy(cluster_col, col_name)
        .count()
        .withColumn("rn_top", F.row_number().over(w_top))
        .filter(F.col("rn_top") == 1)
        .select(
            F.col(cluster_col),
            F.col(col_name).alias(f"top_{col_name}"),
            F.col("count").alias(f"top_{col_name}_n"),
        )
    )


kmeans_numeric_profile = (
    kmeans_train.groupBy("cluster")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
        F.round(F.expr("percentile_approx(trip_distance, 0.5, 1000)"), 2).alias("median_distance_mi"),
        F.round(F.avg("trip_duration_min"), 2).alias("avg_duration_min"),
        F.round(F.expr("percentile_approx(trip_duration_min, 0.5, 1000)"), 2).alias("median_duration_min"),
        F.round(F.avg("average_speed_mph_capped"), 2).alias("avg_speed_capped_mph"),
        F.round(F.avg("passenger_count"), 2).alias("avg_passengers"),
        F.round(F.avg("is_flex_fare"), 4).alias("fraction_flex"),
    )
)

kmeans_profile = kmeans_numeric_profile
for profile_col in ["pu_macro_zone", "RatecodeID", "day_hour_bucket", "cbd_period_flag"]:
    kmeans_profile = kmeans_profile.join(top_category_by_cluster(kmeans_train, profile_col), "cluster", "left")

kmeans_profile.orderBy("cluster").show(truncate=False)

kmeans_train.unpersist()
kmeans_test.unpersist()

+-------+-------+---------------+------------------+----------------+-------------------+--------------------+--------------+-------------+-----------------+-------------------+--------------+----------------+-------------------+---------------------+-------------------+---------------------+
|cluster|n      |avg_distance_mi|median_distance_mi|avg_duration_min|median_duration_min|avg_speed_capped_mph|avg_passengers|fraction_flex|top_pu_macro_zone|top_pu_macro_zone_n|top_RatecodeID|top_RatecodeID_n|top_day_hour_bucket|top_day_hour_bucket_n|top_cbd_period_flag|top_cbd_period_flag_n|
+-------+-------+---------------+------------------+----------------+-------------------+--------------------+--------------+-------------+-----------------+-------------------+--------------+----------------+-------------------+---------------------+-------------------+---------------------+
|0      |542923 |2.76           |2.36              |17.43           |15.37              |9.85                |1.0     

DataFrame[stratum_id: string, VendorID: tinyint, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: tinyint, trip_distance: float, RatecodeID: tinyint, store_and_fwd_flag: string, PULocationID: smallint, DOLocationID: smallint, payment_type: tinyint, fare_amount: float, extra: float, mta_tax: float, tip_amount: float, tolls_amount: float, improvement_surcharge: float, total_amount: float, congestion_surcharge: float, Airport_fee: float, cbd_congestion_fee: float, pu_macro_zone: string, payment_group: string, pickup_hour: int, dow: int, day_hour_bucket: string, trip_distance_bin: string, is_flex_fare: tinyint, cbd_period_flag: string, split_id: string, trip_duration_min: float, average_speed_mph: double, average_speed_mph_capped: double, pu_macro_zone_idx: double, RatecodeID_idx: double, day_hour_bucket_idx: double, pu_macro_zone_ohe: vector, RatecodeID_ohe: vector, day_hour_bucket_ohe: vector, num_features: vector, scaled_num_features: vector, f

In [22]:
GMM_MAX_ITER = 20

gmm = GaussianMixture(
    featuresCol="features",
    predictionCol="gmm_cluster",
    probabilityCol="gmm_probability",
    k=SELECTED_K,
    seed=SEED_M,
    maxIter=GMM_MAX_ITER,
)

gmm_model = gmm.fit(train_features)
gmm_train = gmm_model.transform(train_features).cache()
gmm_test = gmm_model.transform(test_features).cache()

gmm_evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="gmm_cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

gmm_train_profile = gmm_train.withColumn("gmm_prob_max", F.array_max(vector_to_array("gmm_probability")))

gmm_counts = (
    gmm_train_profile.groupBy("gmm_cluster")
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / F.lit(n_train_features), 4).alias("cluster_fraction"),
        F.round(F.avg("gmm_prob_max"), 4).alias("avg_max_probability"),
    )
    .orderBy("gmm_cluster")
)

gmm_metrics = {
    "modelo": "GaussianMixture",
    "k": SELECTED_K,
    "silhouette_train": gmm_evaluator.evaluate(gmm_train.select("features", "gmm_cluster")),
    "silhouette_test": gmm_evaluator.evaluate(gmm_test.select("features", "gmm_cluster")),
    "log_likelihood_train": float(gmm_model.summary.logLikelihood),
}

print(gmm_metrics)
gmm_counts.show(truncate=False)

gmm_train.unpersist()
gmm_test.unpersist()

{'modelo': 'GaussianMixture', 'k': 5, 'silhouette_train': 0.35387036590582543, 'silhouette_test': 0.35446290864690616, 'log_likelihood_train': 135766556.92610013}
+-----------+-------+----------------+-------------------+
|gmm_cluster|count  |cluster_fraction|avg_max_probability|
+-----------+-------+----------------+-------------------+
|0          |192009 |0.0482          |0.9895             |
|1          |2920951|0.7328          |0.9999             |
|2          |624458 |0.1567          |0.9997             |
|3          |61043  |0.0153          |0.9959             |
|4          |187556 |0.0471          |1.0                |
+-----------+-------+----------------+-------------------+



DataFrame[stratum_id: string, VendorID: tinyint, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: tinyint, trip_distance: float, RatecodeID: tinyint, store_and_fwd_flag: string, PULocationID: smallint, DOLocationID: smallint, payment_type: tinyint, fare_amount: float, extra: float, mta_tax: float, tip_amount: float, tolls_amount: float, improvement_surcharge: float, total_amount: float, congestion_surcharge: float, Airport_fee: float, cbd_congestion_fee: float, pu_macro_zone: string, payment_group: string, pickup_hour: int, dow: int, day_hour_bucket: string, trip_distance_bin: string, is_flex_fare: tinyint, cbd_period_flag: string, split_id: string, trip_duration_min: float, average_speed_mph: double, average_speed_mph_capped: double, pu_macro_zone_idx: double, RatecodeID_idx: double, day_hour_bucket_idx: double, pu_macro_zone_ohe: vector, RatecodeID_ohe: vector, day_hour_bucket_ohe: vector, num_features: vector, scaled_num_features: vector, f

In [23]:
clustering_model_results_pdf = pd.DataFrame([
    {
        "modelo": "KMeans",
        "k": SELECTED_K,
        "silhouette_train": final_kmeans_sil_train,
        "silhouette_test": final_kmeans_sil_test,
        "log_likelihood_train": None,
    },
    gmm_metrics,
])

display(clustering_model_results_pdf)

for cached_df in [
    train_features,
    test_features,
    train_cluster_ml,
    test_cluster_ml,
    train_base_ml,
    test_base_ml,
    train_df,
    test_df,
    M_split,
    M,
]:
    try:
        cached_df.unpersist()
    except Exception:
        pass

,modelo,k,silhouette_train,silhouette_test,log_likelihood_train
0,KMeans,5,0.455639,0.455857,NaN
1,GaussianMixture,5,0.353870,0.354463,1.357666e+08


**Lectura del track no supervisado**

El barrido de KMeans muestra un compromiso claro entre separación y balance. `k = 3` tiene la silhouette test más alta (0.4916), pero concentra 71.6% de los viajes en un solo cluster. `k = 6` reduce mucho el cluster dominante (33.8%), pero su silhouette test cae a 0.2408. `k = 5` mantiene silhouette test 0.4559 y baja el cluster dominante a 64.9%, por lo que funciona como solución intermedia entre calidad geométrica e interpretabilidad operativa.

El perfil de KMeans con `k = 5` separa grupos reconocibles para el problema: Flex urbano, viajes urbanos regulares cortos dominados por Manhattan y RatecodeID 1, pasajeros múltiples, viajes medianos más rápidos y viajes largos/aeropuerto con distancia media de 17.56 millas.

La comparación contra `GaussianMixture` favorece a KMeans. GMM alcanza silhouette test 0.3545, menor que KMeans, y concentra 73.3% de los viajes en un componente. Aunque las probabilidades máximas promedio son altas, el resultado es menos útil como segmentación operativa porque un solo componente absorbe demasiada variación.


## 5 Análisis de resultados

### Lectura del modelo supervisado en el problema

El modelo supervisado no busca explicar el cobro total al pasajero, sino estimar `fare_amount` sin usar columnas monetarias derivadas que causarían fuga de información. En ese contexto, el resultado es fuerte: `RandomForestRegressor` alcanza R² 0.9167, lo que significa que distancia, duración, zona, horario, pasajeros y régimen tarifario explican la mayor parte de la tarifa base. En términos del problema, la tarifa de taxi amarillo es altamente predecible cuando el viaje sigue el esquema operativo regular.

La métrica más interpretable es MAE. Un MAE de 1.6890 USD significa que, para un viaje típico del conjunto de prueba, la predicción del bosque se desvía alrededor de 1.69 dólares de la tarifa base real. Operativamente, ese error es suficientemente bajo para estimaciones agregadas, simulaciones de ingresos o monitoreo de patrones normales. El RMSE de 5.1142 USD es mayor porque penaliza más los errores grandes; por eso revela una cola de viajes donde el modelo falla más, no solo el comportamiento promedio.

La diferencia entre Metered y Flex Fare es el hallazgo supervisado más importante. En Metered, el MAE baja a 1.1757 USD, una desviación pequeña para tarifas reguladas por taxímetro. En Flex Fare, el MAE sube a 4.5890 USD y el RMSE a 7.8568 USD. En el mundo real esto sugiere que las tarifas upfront no obedecen exactamente la misma lógica observable con distancia, duración y zona; probablemente incorporan reglas dinámicas o condiciones no disponibles en el dataset. Por eso el área de oportunidad no es solo bajar una métrica, sino tratar Flex como un subproblema: modelo separado, variables adicionales o evaluación específica para ese régimen.

Comparado con Semana 5, M completa produce una lectura más exigente. El RMSE empeora ligeramente y R² baja, pero el MAE mejora. Esto no contradice la entrega anterior: Semana 5 probó el enfoque supervisado sobre M' como muestra manejable; Etapa 3 evalúa la calidad con mayor soporte y más exposición a casos difíciles. La conclusión cualitativa se mantiene: el bosque aleatorio supera al baseline lineal, pero Flex Fare limita el desempeño global.

### Lectura del clustering en el problema

El clustering no predice una tarifa; resume millones de viajes en arquetipos operativos. Por eso silhouette y balance deben leerse junto con los perfiles. Una silhouette test de 0.4559 para KMeans `k = 5` indica separación moderada: los grupos no son clases perfectas, pero sí capturan patrones distinguibles dentro de una ciudad donde muchos viajes comparten zonas, duraciones y distancias parecidas.

El cluster dominante de 64.9% no es necesariamente un error del modelo; refleja que la operación de taxis amarillos está muy concentrada en viajes urbanos regulares, especialmente Manhattan y RatecodeID 1. Sin embargo, sí limita la utilidad analítica: si un grupo contiene casi dos tercios de los viajes, todavía mezcla demasiada operación normal. Por eso `k = 5` se elige como compromiso y no como solución perfecta: separa Flex urbano, pasajeros múltiples, viajes medianos rápidos y viajes largos/aeropuerto sin sacrificar demasiada silhouette.

GaussianMixture queda por debajo para este objetivo. Su silhouette test de 0.3545 y un componente dominante de 73.3% significan que, aunque el modelo asigna probabilidades con confianza, no entrega segmentos tan accionables. En un problema operativo, un cluster útil debe ayudar a describir tipos de viaje; una asignación probabilística segura pero concentrada en un macrogrupo aporta menos para interpretación.

El cap de velocidad a 40 mph también tiene una interpretación práctica. Solo 0.3582% de los viajes de entrenamiento supera ese umbral y el percentil 99 está en 34.51 mph, pero el percentil 99.9 salta a valores no plausibles por duraciones o distancias extremas. Capar la velocidad no cambia el comportamiento típico; evita que errores o casos extremos distorsionen la geometría de KMeans y GMM.

### Efecto de usar M completa

Usar M completa mejora la calidad de la evaluación, no garantiza que cada métrica mejore. En supervisado, M completa hace visible una cola de errores más dura, especialmente Flex Fare. En no supervisado, mejora la separación medida por silhouette frente a Semana 6, pero también muestra con más claridad la dominancia de viajes urbanos regulares. Esa tensión es justamente el valor de esta etapa: con más datos, las conclusiones son más estables y también más honestas sobre los límites del modelo.

La ventaja metodológica es que el split de prueba contiene 1,006,042 filas antes del filtro de duración y 996,458 filas para modelado, con los 240 estratos presentes en train y test. Esto da soporte empírico suficiente para métricas globales, segmentos tarifarios y perfiles raros. Los warnings de Spark durante clustering se explican por presión de caché sobre vectores grandes y múltiples DataFrames persistidos; no invalidan los resultados, pero sí muestran el costo real de evaluar modelos sobre Big Data.

### Relación con entregas anteriores

Los resultados de esta etapa complementan, no sustituyen, las entregas previas. Etapa 2 sigue siendo la fuente de verdad para el diseño muestral; Semana 5 y Semana 6 siguen siendo válidas como aplicaciones de modelos sobre M' y como base para features, pipelines y decisiones de modelado. Etapa 3 toma esas decisiones y las evalúa sobre M completa, con énfasis en métricas, segmentos de error y análisis crítico.

Solo se invalidaría una entrega anterior si esta etapa revelara fuga de información, partición sesgada o construcción incorrecta de M. En esta corrida no ocurre eso: las columnas monetarias derivadas se excluyen del modelo supervisado, train y test son disjuntos, los 240 estratos están presentes en ambos conjuntos y la muestra conserva la estructura definida en Etapa 2.

### Cierre comparativo

El mejor modelo supervisado es `RandomForestRegressor`: sirve para estimar tarifa base con error típico bajo, especialmente en viajes Metered, pero requiere trabajo adicional para Flex Fare. El mejor modelo no supervisado es KMeans con `k = 5`: sirve para describir arquetipos operativos interpretables, aunque el patrón urbano regular sigue dominando la muestra.

En conjunto, las métricas no se quedan como números abstractos. RMSE y MAE traducen el error a dólares por viaje; R² indica qué tan explicable es la tarifa con variables operativas; silhouette y balance indican si los arquetipos realmente separan comportamientos de viaje útiles. Bajo esa lectura, la etapa cumple el objetivo de seleccionar modelos y medir su calidad sobre una muestra representativa y suficientemente grande.


## Declaración de uso de inteligencia artificial

Google. (2026). Gemini 3.5 Flash [Modelo de lenguaje grande], utilizado como apoyo para aprendizaje del contenido, generación y depuración de código, y validación conceptual. https://deepmind.google/models/gemini/flash/